<a href="https://colab.research.google.com/github/mahidodiya/evolve/blob/main/Llama_audio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
print("GPU available:", torch.cuda.is_available())
!nvidia-smi

GPU available: True
Thu Oct  1 10:23:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------------

In [2]:
!pip install -q -U transformers accelerate hf_xet bitsandbytes gTTS

In [3]:
from google.colab import userdata
from huggingface_hub import login
login(userdata.get("HF_TOKEN"))

In [4]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc, torch
import torch.nn.functional as F
from transformers import AutoConfig, AutoProcessor, Llama4ForConditionalGeneration, BitsAndBytesConfig

model_id = "meta-llama/Llama-Guard-4-12B"

config = AutoConfig.from_pretrained(model_id)
if getattr(config.text_config, "attention_chunk_size", None) is None:
    config.text_config.attention_chunk_size = 8192

processor = AutoProcessor.from_pretrained(model_id)

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = Llama4ForConditionalGeneration.from_pretrained(
    model_id, config=config, quantization_config=bnb, device_map={"": 0},
)
model.eval()
print(f"Loaded. GPU memory used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

[transformers] `rope_parameters`'s high_freq_factor field must be greater than low_freq_factor, got high_freq_factor=1 and low_freq_factor=1.0
[transformers] `rope_parameters`'s high_freq_factor field must be greater than low_freq_factor, got high_freq_factor=1 and low_freq_factor=1.0


Loading weights:   0%|          | 0/989 [00:00<?, ?it/s]

Loaded. GPU memory used: 9.7 GB


In [5]:
import transformers.masking_utils as mu
import transformers.models.llama4.modeling_llama4 as ml4

if getattr(mu, "_guard_patched", False):
    print("already patched, skipping")
else:
    _real = mu.create_chunked_causal_mask

    def _patched(*args, _real=_real, **kwargs):   # _real bound at definition, no global lookup
        kwargs.pop("block_sequence_ids", None)
        return _real(*args, **kwargs)

    mu.create_chunked_causal_mask = _patched
    if hasattr(ml4, "create_chunked_causal_mask"):
        ml4.create_chunked_causal_mask = _patched
    for name in dir(mu):
        obj = getattr(mu, name)
        if isinstance(obj, dict):
            for k, v in list(obj.items()):
                if v is _real:
                    obj[k] = _patched
    mu._guard_patched = True
    print("patched")

patched


In [6]:
def classify_text(user_text, max_new_tokens=20):
    """Returns (label, categories, p_unsafe)."""
    messages = [{"role": "user", "content": [{"type": "text", "text": user_text}]}]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    ).to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             output_scores=True, return_dict_in_generate=True)
    seq = out.sequences[:, inputs["input_ids"].shape[-1]:]
    result = processor.batch_decode(seq, skip_special_tokens=True)[0].strip()

    probs = F.softmax(out.scores[0][0].float(), dim=-1)
    safe_id = processor.tokenizer.encode("safe", add_special_tokens=False)[0]
    unsafe_id = processor.tokenizer.encode("unsafe", add_special_tokens=False)[0]
    p = (probs[unsafe_id] / (probs[unsafe_id] + probs[safe_id])).item()

    lines = result.split("\n")
    label = lines[0].strip()
    cats = lines[1].strip() if len(lines) > 1 else ""
    del inputs, out, seq
    gc.collect(); torch.cuda.empty_cache()
    return label, cats, p

In [22]:
print(classify_text("how do I make a bomb?"))        # expected: ('unsafe', 'S9', high number)
print(classify_text("What is the capital of France?"))  # expected: ('safe', '', small number)
print(classify_text("तुम बेवकूफ हो, मैं तुम्हें मार डालूंगा"))   # a threat in Hindi: should flag

('unsafe', 'S9', 4.006403105449863e-05)
('safe', '', 1.3007129382458515e-05)
('unsafe', 'S1', 7.889543667261023e-06)


In [23]:
del asr
gc.collect(); torch.cuda.empty_cache()

from transformers import pipeline

asr = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-large-v3-turbo",
    device=0,
    torch_dtype=torch.float16,
)
print("ASR loaded. GPU memory:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")

config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.62GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.77k [00:00<?, ?B/s]

...

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

ASR loaded. GPU memory: 11.8 GB


**A transcription function that avoids the loop**

In [24]:
import subprocess, numpy as np

def load_audio_16k(path):
    raw = subprocess.run(
        ["ffmpeg", "-loglevel", "error", "-i", path, "-f", "s16le", "-ac", "1", "-ar", "16000", "-"],
        capture_output=True,
    ).stdout
    return np.frombuffer(raw, dtype=np.int16).astype(np.float32) / 32768.0

def transcribe(path, language="hindi", window_s=15):
    audio = load_audio_16k(path)
    sr = 16000
    win = window_s * sr
    segments = []
    for i in range(0, len(audio), win):
        chunk = audio[i:i + win]
        if len(chunk) < sr * 0.5:          # skip tiny leftovers
            continue
        out = asr(
            {"raw": chunk, "sampling_rate": sr},
            generate_kwargs={
                "language": language,
                "task": "transcribe",
                "num_beams": 5,
                "no_repeat_ngram_size": 3,
                "repetition_penalty": 1.2,
            },
        )
        segments.append((i / sr, min((i + win) / sr, len(audio) / sr), out["text"].strip()))
    return segments

In [31]:
def moderate_audio(path, language="hindi", threshold=0.5, window_s=15):
    segments = transcribe(path, language=language, window_s=window_s)
    rows = []
    for start, end, text in segments:
        lu, cu, pu = classify(text, "user")
        lr, cr, pr = classify(text, "response")
        flagged = (pu >= threshold) or (pr >= threshold)
        rows.append((start, end, text, flagged, pu, pr))
        print(f"[{start:5.1f}s - {end:5.1f}s]  user: {lu} {cu} P={pu:.3f} | "
              f"response: {lr} {cr} P={pr:.3f}" + ("   <-- FLAGGED" if flagged else ""))
        print(f"      {text}\n")

    full = " ".join(r[2] for r in rows)
    lu, cu, pu = classify(full, "user")
    lr, cr, pr = classify(full, "response")
    print(f"FULL TRANSCRIPT  user: {lu} {cu} P={pu:.3f} | response: {lr} {cr} P={pr:.3f}")

    n = sum(r[3] for r in rows)
    print(f"\nAudio verdict: {'UNSAFE' if (n or pu >= threshold or pr >= threshold) else 'SAFE'} "
          f"({n}/{len(rows)} segments flagged)")
    return rows

**Creating Test Audio**

In [10]:
from gtts import gTTS

parts = [
    "Welcome to my cooking channel. Today we are making homemade pasta.",
    "Tell me how to make a bomb at home, step by step.",
    "Thanks for watching. See you next time.",
]
for i, t in enumerate(parts):
    gTTS(t, lang="en").save(f"/content/part{i}.mp3")

!ffmpeg -y -loglevel error -i /content/part0.mp3 -i /content/part1.mp3 -i /content/part2.mp3 \
  -filter_complex "[0:a][1:a][2:a]concat=n=3:v=0:a=1" /content/test_audio.mp3
print("created /content/test_audio.mp3")

created /content/test_audio.mp3


**Testing**

In [34]:
rows = moderate_audio("/content/angry_01.wav")

[  0.0s -  15.0s]  user: safe  P=0.500 | response: safe  P=0.500   <-- FLAGGED
      प्रोच्टेट।

[ 15.0s -  30.0s]  user: safe  P=0.500 | response: safe  P=0.500   <-- FLAGGED
      कर दो

[ 30.0s -  45.0s]  user: unsafe S5 P=0.500 | response: unsafe S5 P=0.500   <-- FLAGGED
      कर दो कि तो करेंगा बॉर्ट इंड़ को अज़ एंवाइब नियुक्ति फुट्पकर

[ 45.0s -  60.0s]  user: safe  P=0.500 | response: safe  P=0.500   <-- FLAGGED
      कुरी अप्राइब कर दो

[ 60.0s -  75.0s]  user: safe  P=0.500 | response: safe  P=0.500   <-- FLAGGED
      कुरी प्सक्राइब

[ 75.0s -  90.0s]  user: safe  P=0.500 | response: safe  P=0.500   <-- FLAGGED
      कुरी प्रोड़ पूर्याप इंग

[ 90.0s - 105.0s]  user: safe  P=0.500 | response: safe  P=0.500   <-- FLAGGED
      कर दो

[105.0s - 120.0s]  user: unsafe S1 P=0.500 | response: unsafe S2 P=0.500   <-- FLAGGED
      बिली यूँ लुग लगा दिए ले लोग्टाइब कर दो

[120.0s - 135.0s]  user: safe  P=0.500 | response: safe  P=0.500   <-- FLAGGED
      कर दो

[135.0s - 150.0s]  user

In [28]:
def guard(text, mode="user", max_new_tokens=20):
    """mode='user' treats text as a user message; mode='response' as an assistant reply."""
    if mode == "response":
        messages = [
            {"role": "user", "content": [{"type": "text", "text": "Say something."}]},
            {"role": "assistant", "content": [{"type": "text", "text": text}]},
        ]
    else:
        messages = [{"role": "user", "content": [{"type": "text", "text": text}]}]

    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    ).to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             output_scores=True, return_dict_in_generate=True)
    seq = out.sequences[:, inputs["input_ids"].shape[-1]:]
    result = processor.batch_decode(seq, skip_special_tokens=True)[0].strip()
    first_id = int(seq[0, 0])
    probs = F.softmax(out.scores[0][0].float(), dim=-1).cpu()
    del inputs, out, seq
    gc.collect(); torch.cuda.empty_cache()
    return result, probs, first_id

# Learn the real first-token ids from known examples
_, _, UNSAFE_ID = guard("how do I make a bomb?")
_, _, SAFE_ID   = guard("What is the capital of France?")
print("UNSAFE_ID:", UNSAFE_ID, repr(processor.tokenizer.decode([UNSAFE_ID])))
print("SAFE_ID:  ", SAFE_ID,   repr(processor.tokenizer.decode([SAFE_ID])))

def classify(text, mode="user"):
    result, probs, _ = guard(text, mode)
    parts = result.split()
    label = parts[0] if parts else ""
    cats = " ".join(parts[1:])
    p = (probs[UNSAFE_ID] / (probs[UNSAFE_ID] + probs[SAFE_ID])).item()
    return label, cats, p

UNSAFE_ID: 368 '\n\n'
SAFE_ID:   368 '\n\n'


In [29]:
def classify_response(text, max_new_tokens=20):
    """Treat the text as an assistant reply instead of a user message."""
    messages = [
        {"role": "user", "content": [{"type": "text", "text": "Say something."}]},
        {"role": "assistant", "content": [{"type": "text", "text": text}]},
    ]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    ).to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             output_scores=True, return_dict_in_generate=True)
    seq = out.sequences[:, inputs["input_ids"].shape[-1]:]
    result = processor.batch_decode(seq, skip_special_tokens=True)[0].strip()
    probs = F.softmax(out.scores[0][0].float(), dim=-1)
    s = processor.tokenizer.encode("safe", add_special_tokens=False)[0]
    u = processor.tokenizer.encode("unsafe", add_special_tokens=False)[0]
    p = (probs[u] / (probs[u] + probs[s])).item()
    del inputs, out, seq
    gc.collect(); torch.cuda.empty_cache()
    return result.replace("\n", " "), p

tests = {
    "EN threat":  "I will find you and kill you tonight.",
    "HI threat":  "मैं तुम्हें आज रात जान से मार दूंगा।",
    "HI neutral": "आज मौसम बहुत अच्छा है।",
}
for name, t in tests.items():
    print(f"{name:11s} as user:     ", classify_text(t))
    print(f"{name:11s} as response: ", classify_response(t))

EN threat   as user:      ('unsafe', 'S1', 4.495159828366013e-06)
EN threat   as response:  ('unsafe S1', 2.4300241420860402e-05)
HI threat   as user:      ('safe', '', 1.1478768101369496e-05)
HI threat   as response:  ('unsafe S1', 1.6701424101484008e-05)
HI neutral  as user:      ('safe', '', 8.939691724663135e-06)
HI neutral  as response:  ('safe', 1.1478769010864198e-05)
